# Metodologia 1: segmentação 2D e reconstrução 3D

Exemplo executável da metodologia de segmentação 2D e reconstrução 3D: pré-processar um volume, segmentar seus planos independentemente e reconciliar os identificadores entre planos vizinhos. Não há treinamento de modelo.

O roteiro chama as funções existentes, sem manter cópias dos algoritmos no notebook. Primeiro percorremos as etapas explicitamente; depois verificamos que `SliceWiseFiberPipeline` produz os mesmos resultados. As medidas são calculadas **após** a reconstrução 3D.

**Execução:** rode as células em ordem, com o kernel do ambiente científico do projeto. Usamos um cubo de 200 × 200 × 200 voxels de `data/PPGF40.tif`, sem salvar resultados automaticamente. O arquivo precisa estar disponível localmente. Os antigos resultados exploratórios não são reutilizados.

## 1. Preparar o ambiente

São necessários NumPy, SciPy, scikit-image, skan, pandas, Matplotlib, tifffile, tqdm e Jupyter. O código 2D atual requer **scikit-image 0.26 ou posterior**. Abra o notebook a partir da raiz do repositório ou de `notebooks/`.

As classes recebem arrays já selecionados; leitura, normalização, ROI, medidas e exportação ficam separadas. A ordem dos eixos dos arrays é sempre **(Z, Y, X)**.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tifffile
from IPython.display import display
from scipy.ndimage import binary_opening
from skimage.color import label2rgb
from skimage.morphology import skeletonize

root = Path.cwd()
if not (root / "fiber_analysis").is_dir():
    root = root.parent
if not (root / "fiber_analysis").is_dir():
    raise FileNotFoundError("Abra o notebook na raiz do repositório ou em notebooks/.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from fiber_analysis.config import SliceWiseConfig
from fiber_analysis.metrics import measure_fibers
from fiber_analysis.pipelines import SliceWiseFiberPipeline, skeleton_with_red_junctions
from src_2d.utils.preprocessing import binarize_volume_otsu, select_cube_region
from src_2d.utils.reconstruction import propagate_labels
from src_2d.utils.segmentation import segment_fibers_2d, segment_volume_slicewise

## 2. Escolher e carregar o volume

Mapeamos `data/PPGF40.tif` em memória, em modo somente leitura, e selecionamos um cubo central de 200 voxels por aresta dentro da ROI da amostra. Não há entrada sintética ou fallback para arquivo ausente.

Preservamos a normalização histórica: dividir pelo **máximo do TIFF inteiro**, multiplicar por 255 e converter para `uint8`. A busca do máximo percorre os dados, mas a conversão é feita somente no cubo selecionado, sem criar um volume completo em ponto flutuante. Isso produz os mesmos valores que normalizar o TIFF inteiro antes do recorte.

A ROI `[:, 150:820, 129:491]` pertence especificamente a essa amostra. Ajuste-a se trocar o TIFF. A seleção do cubo central de 200 voxels ocorre antes da classe e não faz parte da segmentação.

In [ ]:
arquivo_tiff = root / "data" / "PPGF40.tif"
aresta_cubo = 200

volume_completo = tifffile.memmap(arquivo_tiff, mode="r")
intensidade_maxima = volume_completo.max()
roi = volume_completo[:, 150:820, 129:491]
cubo = select_cube_region(roi, edge_length=aresta_cubo, position="center")
volume_selecionado = (cubo / intensidade_maxima * 255).astype(np.uint8)
del volume_completo, roi, cubo

print(f"Entrada: {arquivo_tiff}")
print(f"Volume selecionado (Z, Y, X): {volume_selecionado.shape}")
print(f"Tipo: {volume_selecionado.dtype}")

## 3. Centralizar os parâmetros e identificar o plano

Usamos os padrões de `SliceWiseConfig`, explicitados abaixo. `local_range=12` é um deslocamento em pixels ao longo do ramo, **não** o tamanho de uma janela SVD 3D.

**Convenção histórica preservada:** os nomes `x`, `y`, `z` selecionam as dimensões 0, 1, 2 do array, respectivamente. Assim, `slicing_axis='z'` percorre `volume[:, :, i]`: planos **YZ ao longo de X**, considerando arrays ZYX. Isso difere da descrição de planos XY no texto metodológico; não corrigimos silenciosamente a convenção do código.

O corte central será usado apenas para visualizar as etapas; Otsu e abertura continuam sendo calculados no volume inteiro selecionado.

In [ ]:
config = SliceWiseConfig(
    slicing_axis="z",
    max_angle_deviation=45,
    remove_small_skeletons=True,
    min_skeleton_size=20,
    min_branch_length=10,
    local_analysis=True,
    local_range=12,
    propagation_min_overlap=1,
    propagation_min_overlap_ratio=0.0,
)
parametros_2d = {
    nome: getattr(config, nome)
    for nome in (
        "max_angle_deviation",
        "remove_small_skeletons",
        "min_skeleton_size",
        "min_branch_length",
        "local_analysis",
        "local_range",
    )
}
dim_corte = {"x": 0, "y": 1, "z": 2}[config.slicing_axis]
indice_corte = volume_selecionado.shape[dim_corte] // 2
eixo_horizontal, eixo_vertical = (("X", "Y"), ("X", "Z"), ("Y", "Z"))[dim_corte]
corte_original = np.take(volume_selecionado, indice_corte, axis=dim_corte)
print(config)
print(f"Inspeção: dimensão {dim_corte}, índice {indice_corte}")

## 4. Pré-processar: Otsu global e abertura 3D

`binarize_volume_otsu` estima um único limiar para o volume selecionado. Voxels **acima** dele são foreground. `binary_opening` aplica erosão seguida de dilatação com a estrutura padrão da SciPy, reduzindo pequenas protrusões.

A abertura ocorre em **3D antes do fatiamento**, exatamente como na classe e no runner. Não aplicamos mediana, fechamento, limiar manual ou uma abertura 2D adicional. O painel abaixo é somente um corte da máscara volumétrica.

In [ ]:
mascara_otsu, limiar = binarize_volume_otsu(volume_selecionado)
mascara = binary_opening(mascara_otsu)
corte_mascara = np.take(mascara, indice_corte, axis=dim_corte)
print(f"Limiar de Otsu: {limiar}; voxels de foreground: {int(mascara.sum())}")

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].hist(volume_selecionado.ravel(), bins=64, color="gray")
axes[0].axvline(limiar, color="red", label=f"Otsu = {limiar}")
axes[0].set(xlabel="Intensidade", ylabel="Voxels", title="Histograma do volume")
axes[0].legend()
for ax, imagem, titulo in zip(
    axes[1:], (corte_original, corte_mascara), ("Tons de cinza", "Após abertura 3D")
):
    ax.imshow(imagem, cmap="gray")
    ax.set(title=titulo, xlabel=eixo_horizontal, ylabel=eixo_vertical)
fig.tight_layout()
plt.show()

## 5. Entender a segmentação de um plano

`segment_fibers_2d` executa, nesta ordem:

1. Esqueletização da máscara 2D.
2. Remoção de ramos curtos JP–EP e EP–EP; JP–JP não é removido por comprimento. JP é junção e EP é extremidade; a junção é preservada ao remover um ramo terminal.
3. Estimativa da direção local e poda em cada junção: mantém o melhor par somente se seu desvio em relação a 180° for **estritamente menor** que `max_angle_deviation`. O valor 45 não é um ângulo mínimo entre ramos.
4. Componentes conexos com conectividade 8, seguidos da remoção de marcadores com menos de `min_skeleton_size` pixels, quando habilitada.
5. Watershed sobre o negativo da distância euclidiana, usando os componentes do esqueleto como marcadores e a máscara binária como limite.

O esqueleto inicial abaixo é recalculado apenas para visualização. As outras duas imagens são os resultados da função existente: marcadores após filtragem/poda e regiões expandidas pelo watershed.

In [ ]:
esqueleto_inicial_2d = skeletonize(corte_mascara)
fibras_2d, marcadores_2d = segment_fibers_2d(corte_mascara, **parametros_2d)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
imagens = (
    corte_mascara,
    esqueleto_inicial_2d,
    label2rgb(marcadores_2d, bg_label=0),
    label2rgb(fibras_2d, bg_label=0),
)
titulos = (
    "Máscara",
    "Esqueleto inicial",
    "Marcadores filtrados/podados",
    "Watershed 2D",
)
for ax, imagem, titulo in zip(axes, imagens, titulos):
    ax.imshow(imagem, cmap="gray")
    ax.set(title=titulo, xlabel=eixo_horizontal, ylabel=eixo_vertical)
fig.tight_layout()
plt.show()

## 6. Segmentar todos os planos independentemente

`segment_volume_slicewise` aplica a mesma função anterior ao longo da dimensão escolhida. O resultado tem a forma do volume original, mas seus IDs são **locais a cada plano**: o ID 1 em dois planos ainda não significa a mesma fibra.

Também produzimos o esqueleto 3D diagnóstico usado pela classe. Seus voxels com pelo menos três vizinhos na vizinhança 26 são marcados em vermelho no RGB. **Esse esqueleto 3D não é usado como marcador do watershed 2D**; não confundir o diagnóstico com os esqueletos planares.

In [ ]:
esqueleto_3d, juncoes_3d, esqueleto_rgb = skeleton_with_red_junctions(mascara)
rotulos_planares, marcadores_planares = segment_volume_slicewise(
    mascara, axis=config.slicing_axis, verbose=False, **parametros_2d
)
np.testing.assert_array_equal(
    np.take(rotulos_planares, indice_corte, axis=dim_corte), fibras_2d
)
np.testing.assert_array_equal(
    np.take(marcadores_planares, indice_corte, axis=dim_corte), marcadores_2d
)
print(f"Rótulos planares: {rotulos_planares.shape}, {rotulos_planares.dtype}")
print(f"Esqueleto 3D diagnóstico: {int(esqueleto_3d.sum())} voxels")

## 7. Reconstruir identidades 3D por sobreposição

`propagate_labels` percorre planos consecutivos. Cada região procura o predecessor de maior sobreposição. A herança exige `min_overlap` pixels e uma razão `sobreposição / menor área` de pelo menos `min_overlap_ratio`.

Quando duas regiões disputam o mesmo predecessor, apenas a de maior sobreposição absoluta herda o ID. As demais recebem novos IDs; a propagação não modela bifurcações nem procura correspondências através de planos vazios.

Assim como na classe, **não passamos `foreground_mask`**: foreground sem marcador não é recuperado automaticamente. Só depois desta etapa os rótulos representam identidades globais. Ainda podem ocorrer fragmentação ou fusão de fibras.

In [ ]:
rotulos_globais = propagate_labels(
    rotulos_planares,
    axis=config.slicing_axis,
    min_overlap=config.propagation_min_overlap,
    min_overlap_ratio=config.propagation_min_overlap_ratio,
    verbose=False,
)
ids_globais = np.unique(rotulos_globais)
print(f"Instâncias 3D detectadas: {np.count_nonzero(ids_globais)}")

indices_exibidos = sorted(
    {indice_corte, min(indice_corte + 1, mascara.shape[dim_corte] - 1)}
)
fig, axes = plt.subplots(
    len(indices_exibidos), 2, figsize=(9, 4 * len(indices_exibidos)), squeeze=False
)
for linha, indice in enumerate(indices_exibidos):
    for ax, rotulos, titulo in zip(
        axes[linha], (rotulos_planares, rotulos_globais), ("IDs locais", "IDs globais")
    ):
        ax.imshow(
            np.take(rotulos, indice, axis=dim_corte),
            cmap="nipy_spectral",
            vmin=0,
            vmax=int(rotulos.max()),
        )
        ax.set(
            title=f"{titulo}: plano {indice}",
            xlabel=eixo_horizontal,
            ylabel=eixo_vertical,
        )
fig.tight_layout()
plt.show()

## 8. Usar a classe no código de aplicação

Em uma aplicação, **esta chamada substitui as etapas volumétricas 4–7**. Ela recebe os tons de cinza, não a máscara já pré-processada. Não é necessário repetir o processamento manual.

Somente neste roteiro executamos ambas as formas para verificar a equivalência exata dos intermediários e dos rótulos finais. `SliceWiseResult` dá nomes aos arrays, evitando tuplas posicionais; não contém as medidas, que serão calculadas separadamente.

`verbose=True` mostra configuração, etapas numeradas, contagens, tempos e barras por plano. Use `verbose=False` para desativar as mensagens de progresso e as barras internas, sem ocultar avisos ou exceções.

In [ ]:
pipeline = SliceWiseFiberPipeline(config, verbose=True)
resultado = pipeline.segment(volume_selecionado)
for obtido, esperado in (
    (resultado.raw_volume, volume_selecionado),
    (resultado.binary_mask, mascara),
    (resultado.skeleton_3d, esqueleto_3d),
    (resultado.junction_mask, juncoes_3d),
    (resultado.skeleton_rgb, esqueleto_rgb),
    (resultado.slicewise_labels, rotulos_planares),
    (resultado.labels, rotulos_globais),
):
    np.testing.assert_array_equal(obtido, esperado)
print("Classe e sequência explícita: todos os arrays são iguais.")

## 9. Medir comprimento e orientação após a reconstrução

`measure_fibers` reesqueletiza cada rótulo 3D em sua caixa delimitadora. O comprimento `ep_ep_length` é a maior distância euclidiana entre endpoints do esqueleto, **não** o comprimento do caminho. Com menos de dois endpoints, usa a extensão projetada no eixo principal SVD. Casos degenerados mantêm o comportamento histórico de comprimento zero e direção padrão X.

A direção vem do mesmo par de endpoints, ou do eixo SVD no fallback. Os ângulos são `acos(abs(componente))`, entre 0° e 90°: 0° indica alinhamento com o eixo. `voxel_count` conta voxels do **esqueleto**, não o volume da fibra.

`voxel_size=(dz, dy, dx)` deve conter o espaçamento calibrado. `(1, 1, 1)` produz comprimentos em unidades de voxel; não aplicamos conversão fixa para micrômetros. Não há filtro IQR ou outra exclusão automática. Cada instância tem o mesmo peso nos histogramas.

In [ ]:
voxel_size = (1.0, 1.0, 1.0)
registros = measure_fibers(resultado.labels, voxel_size=voxel_size)
medidas = pd.DataFrame(registros)
display(medidas.head(10))

if medidas.empty:
    print("Nenhuma instância medida; histogramas não gerados.")
else:
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    axes[0].hist(medidas["ep_ep_length"], bins=20, color="steelblue")
    axes[0].set(
        title="Comprimento EP–EP", xlabel="Unidade do espaçamento", ylabel="Instâncias"
    )
    for ax, eixo in zip(axes[1:], ("x", "y", "z")):
        ax.hist(
            medidas[f"angle_{eixo}"], bins=np.linspace(0, 90, 19), color="steelblue"
        )
        ax.set(
            title=f"Orientação em relação a {eixo.upper()}",
            xlabel="Ângulo (graus)",
            ylabel="Instâncias",
        )
    fig.tight_layout()
    plt.show()

## 10. Exportar somente quando solicitado

Mantenha `salvar_resultados=False` para não escrever arquivos. Se habilitado, o exemplo salva os rótulos 3D em TIFF e chama `analyze_and_save` para recalcular as mesmas medidas, gravar CSV e gerar os histogramas padronizados. Estes últimos mostram percentuais de instâncias nas orientações; a célula anterior usa contagens.

A pasta é criada apenas ao habilitar a exportação; executar novamente substitui os arquivos de mesmo nome. Sem instâncias, o relatório produz somente o cabeçalho CSV e não gera PNGs. O import de reporting fica aqui porque seleciona o backend Agg para exportação, após as visualizações interativas.

In [ ]:
salvar_resultados = False
pasta_saida = root / "results" / "notebook_examples" / "slice_wise"

if salvar_resultados:
    from fiber_analysis.reporting import analyze_and_save

    pasta_saida.mkdir(parents=True, exist_ok=True)
    tifffile.imwrite(
        pasta_saida / "labels.tif",
        resultado.labels,
        photometric="minisblack",
        metadata={"axes": "ZYX"},
    )
    relatorio = analyze_and_save(
        resultado.labels, pasta_saida, prefix="slice_wise", voxel_size=voxel_size
    )
    print(f"Resultados exportados para: {pasta_saida}")
else:
    print("Exportação desativada; nenhum arquivo foi salvo.")

## Interpretação e próximos roteiros

Um rótulo é uma **instância ou fragmento detectado**, não garantia de uma fibra física completa. Erros de poda planar podem ser propagados; erros de correspondência podem fragmentar ou fundir identidades. Pixels sem marcador podem permanecer sem rótulo. A inspeção do cubo não substitui uma referência anotada para avaliar a precisão dessas decisões.

As implementações estão em `src_2d/utils/segmentation.py` e `src_2d/utils/reconstruction.py`. [fiber_seg_3D.ipynb](fiber_seg_3D.ipynb) mostra a alternativa diretamente tridimensional. [hermans_slice_yz.ipynb](hermans_slice_yz.ipynb) contém a análise de Hermans, com métricas PCA próprias que não são substituídas pelas medidas EP–EP deste roteiro.